# 53-Qubit Google Circuit: Tetron MBQC vs Direct Gate (Clifford / Stabilizer)

This notebook validates the 53-qubit tetron MBQC translation of the Google Sycamore circuit after projecting the circuit to a Clifford-only gate set.

### Clifford projection used here

For an efficient stabilizer simulation, this notebook keeps :

- single-qubit **√X** gates, `X**0.5`;
- single-qubit **√Y** gates, `Y**0.5`;
- two-qubit **fSim(π/2, 0) = iSWAP†**, replacing every `FSimGate(theta, phi)` by this Clifford two-qubit gate.

All `Rz`, `PhasedXPowGate` / √W, and other non-kept gates are dropped. Therefore the comparison is between the **Clifford-projected direct circuit** and the **Clifford-projected tetron MBQC circuit**, not the full calibrated Google circuit.

### Main validation method

The 12-qubit notebook can compare statevectors directly, but a literal 53-qubit statevector has $2^{53}$amplitudes, and the tetron circuit would have 106 physical qubits. Instead of estimating the full output distribution from shots, this notebook uses an exact stabilizer-compatible check:

1. build the MBQC circuit without final data measurements;
2. build the direct Clifford circuit without final measurements;
3. append the inverse direct circuit onto the 53 data tetrons;
4. measure the 53 data tetrons.

If the MBQC translation implements the same Clifford unitary as the direct circuit, every MBQC measurement trajectory should end as

$
U_{\rm direct}^{\dagger} |\psi_{\rm MBQC}^{(s)}\rangle = |0\rangle^{\otimes 53}.
$

The stabilizer simulator still uses multiple shots, but each shot is only a different MBQC measurement trajectory. The result should be deterministically all zeros, not an estimate of the full 53-qubit output distribution.


In [ ]:
import os, sys, importlib.util
import time

import numpy as np
import matplotlib.pyplot as plt
import cirq

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr
from qiskit.compiler import transpile
from qiskit_aer import AerSimulator

# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------
REPO_ROOT  = os.path.abspath('.')
TETRON_DIR = os.path.join(REPO_ROOT, 'src', 'tetron')
GOOGLE_DIR = os.path.join(REPO_ROOT, 'google_53qubits_circuit')

# Make the notebook robust if it is run from the upload/download folder.
for p in [REPO_ROOT, TETRON_DIR, '/mnt/data']:
    if p and os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from qubit_mapping_53 import (
        grid_to_qiskit_index,
        grid_to_sq_ancilla_index,
        grid_edge_to_qiskit_indices,
        GRID_TO_LOGICAL,
    )
from qubit_mapping_53 import (
    grid_to_qiskit_index,
    grid_to_sq_ancilla_index,
    grid_edge_to_qiskit_indices,
    GRID_TO_LOGICAL,
)

print('Imports OK.  Tetron dir:', TETRON_DIR)
print('Google dir: ', GOOGLE_DIR)


Imports OK.  Tetron dir: e:\git_repo\mbqc-circuit-comparison\src\tetron
Google dir:  e:\git_repo\mbqc-circuit-comparison\google_53qubits_circuit


## 2. MBQC helper functions

These are the parity-measurement primitives and MBQC gate builders validated
in `Two_qubit_gate_supremacy.ipynb`.  They all share a single **8-bit scratch
register** — each function writes its measurement outcomes to specific bit
positions, applies the Pauli corrections immediately, then returns (so those
bits can be safely overwritten by the next call).

Bit-position layout of the scratch register:
- `scratch[0:5]` — single-qubit gates (H, S, SH, HS, HSH)
- `scratch[5:8]` — MBQC CNOT (non-overlapping with single-qubit slots)

In [31]:
# ---------------------------------------------------------------------------
# Parity-measurement primitives
# Convention: Y = S X S†  (consistent with Two_qubit_gate_supremacy.ipynb)
# ---------------------------------------------------------------------------

def measure_ZZ(qc, q0, q1, cbit):
    qc.cx(q0, q1)
    qc.measure(q1, cbit)
    qc.cx(q0, q1)

def measure_XI(qc, q0, q1, cbit):
    qc.h(q0)
    qc.measure(q0, cbit)
    qc.h(q0)

def measure_YI(qc, q0, q1, cbit):
    qc.sdg(q0)
    measure_XI(qc, q0, q1, cbit)
    qc.s(q0)

def measure_ZY(qc, q0, q1, cbit):
    qc.sdg(q1)
    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)
    qc.s(q1)

def measure_ZX(qc, q0, q1, cbit):
    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)

In [32]:
# ---------------------------------------------------------------------------
# Single-qubit MBQC gates
# ---------------------------------------------------------------------------

def add_H(qc, d, a, cbit):
    c = cbit
    measure_XI(qc, a, d, c[0])
    measure_ZY(qc, a, d, c[1])
    measure_YI(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])
    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.y(d)
    qc.x(d)
    qc.reset(a)
    return qc

def add_S(qc, d, a, cbit):
    c = cbit
    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_YI(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])
    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.z(d)
    qc.reset(a)
    return qc

def add_SH(qc, d, a, cbit):
    c = cbit
    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_ZY(qc, a, d, c[2])
    measure_YI(qc, a, d, c[3])
    measure_XI(qc, a, d, c[4])
    parity_023 = expr.bit_xor(expr.bit_xor(c[0], c[2]), c[3])
    with qc.if_test(parity_023):
        qc.y(d)
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(parity_12):
        qc.z(d)
    qc.reset(a)
    return qc

def add_HS(qc, d, a, cbit):
    c = cbit
    measure_XI(qc, a, d, c[0])
    measure_ZY(qc, a, d, c[1])
    measure_ZZ(qc, a, d, c[2])
    measure_YI(qc, a, d, c[3])
    measure_XI(qc, a, d, c[4])
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(d)
    parity_013 = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[3])
    with qc.if_test(expr.logic_not(parity_013)):
        qc.z(d)
    qc.reset(a)
    return qc

def add_HSH(qc, d, a, cbit):
    c = cbit
    measure_XI(qc, a, d, c[0])
    measure_ZZ(qc, a, d, c[1])
    measure_ZY(qc, a, d, c[2])
    measure_XI(qc, a, d, c[3])
    parity_03 = expr.bit_xor(c[0], c[3])
    with qc.if_test(parity_03):
        qc.y(d)
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(d)
    qc.reset(a)
    return qc

# Supremacy single-qubit gates
def add_sqrtX(qc, d, a, cbit):
    return add_HSH(qc, d, a, cbit)

def add_sqrtY(qc, d, a, cbit):
    qc = add_S(qc, d, a, cbit)
    qc = add_HS(qc, d, a, cbit)
    return qc

In [33]:
# ---------------------------------------------------------------------------
# MBQC CNOT  —  uses scratch[5], scratch[6], scratch[7]
# (these slots are separate from the single-qubit slots [0:5])
# ---------------------------------------------------------------------------

def add_CNOT(qc, ctrl, anc, targ, cbit):
    qc.reset(anc)
    measure_ZX(qc, ctrl, anc, cbit[5])
    measure_ZX(qc, anc,  targ, cbit[6])
    qc.h(anc)
    qc.measure(anc, cbit[7])
    qc.h(anc)
    qc.reset(anc)
    with qc.if_test(expr.bit_xor(cbit[5], cbit[7])):
        qc.x(targ)
    with qc.if_test((cbit[6], 1)):
        qc.z(ctrl)
    return qc


# ---------------------------------------------------------------------------
# fSim(pi/2, 0) = iSWAP-dagger  (Clifford gate)
#
# Decomposition (from Two_qubit_gate_supremacy.ipynb):
#   fSim(theta, phi) = iSWAPdg(theta) * CPhase(phi)
#   phi = 0  =>  CPhase(0) = I  =>  fSim(pi/2, 0) = iSWAPdg(theta=pi/2)
#
# iSWAPdg(pi/2) circuit:
#   H_c  H_t  CNOT  S_t  CNOT  H_c  H_t  (S^3)_c  (S^3)_t
#   H_c  H_t  CNOT  S_t  CNOT  SH_c  SH_t
# where S^3 = S-dagger and Rz(pi/2) -> S (Clifford)
# ---------------------------------------------------------------------------

def add_fsim_clifford(qc, ctrl, anc, targ, cbit):
    # First half of iSWAPdg
    qc = add_H(qc, ctrl, anc, cbit)
    qc = add_H(qc, targ, anc, cbit)
    qc = add_CNOT(qc, ctrl, anc, targ, cbit)
    qc.s(targ)                        # Rz(pi/2) = S  (Clifford, applied directly)
    qc = add_CNOT(qc, ctrl, anc, targ, cbit)
    qc = add_H(qc, ctrl, anc, cbit)
    qc = add_H(qc, targ, anc, cbit)
    # S-dagger on both qubits, implemented as S*S*S (three S gates)
    for _ in range(3):
        qc = add_S(qc, ctrl, anc, cbit)
    for _ in range(3):
        qc = add_S(qc, targ, anc, cbit)
    # Second half
    qc = add_H(qc, ctrl, anc, cbit)
    qc = add_H(qc, targ, anc, cbit)
    qc = add_CNOT(qc, ctrl, anc, targ, cbit)
    qc.s(targ)                        # Rz(pi/2) = S
    qc = add_CNOT(qc, ctrl, anc, targ, cbit)
    qc = add_SH(qc, ctrl, anc, cbit)
    qc = add_SH(qc, targ, anc, cbit)
    return qc

## 3. Load the Google 53-qubit circuit

In [34]:
CIRCUIT_BASENAME = 'circuit_n53_m12_s0_e0_pABCDCDAB.py'


def find_existing_file(candidates):
    for path in candidates:
        if path and os.path.exists(path):
            return path
    raise FileNotFoundError(
        'Could not find the Google circuit file. Tried:\n' +
        '\n'.join(f'  - {p}' for p in candidates)
    )


CIRCUIT_FILE = find_existing_file([
    os.path.join(GOOGLE_DIR, CIRCUIT_BASENAME),
    os.path.join(REPO_ROOT, CIRCUIT_BASENAME),
    os.path.join('/mnt/data', CIRCUIT_BASENAME),
])


def load_cirq_circuit(path):
    spec = importlib.util.spec_from_file_location('google_circuit', path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.QUBIT_ORDER, mod.CIRCUIT


QUBIT_ORDER, CIRCUIT = load_cirq_circuit(CIRCUIT_FILE)
print(f'Loaded  : {CIRCUIT_FILE}')
print(f'Qubits  : {len(QUBIT_ORDER)}')
print(f'Moments : {len(CIRCUIT)}')


Loaded  : circuit_n53_m12_s0_e0_pABCDCDAB.py
Qubits  : 53
Moments : 49


## 4. Gate classification

Of the gates that appear in the file:
- **√X** (`X**0.5`) → `add_sqrtX`
- **√Y** (`Y**0.5`) → `add_sqrtY`
- **FSimGate(θ, φ)** → `add_fsim_clifford` (replacing all with fSim(π/2, 0))
- **√W** (`PhasedXPowGate`), **Rz**, and anything else → **dropped**

In [35]:
def _is_sqrt_X(g):
    return isinstance(g, cirq.XPowGate) and np.isclose(g.exponent, 0.5)

def _is_sqrt_Y(g):
    return isinstance(g, cirq.YPowGate) and np.isclose(g.exponent, 0.5)

def _is_fsim(g):
    return isinstance(g, cirq.FSimGate)

# Audit gate types in the loaded circuit
gate_types = {}
for moment in CIRCUIT:
    for op in moment.operations:
        t = type(op.gate).__name__
        gate_types[t] = gate_types.get(t, 0) + 1

print('Gate type counts in the loaded circuit:')
for k, v in sorted(gate_types.items()):
    print(f'  {k}: {v}')

kept    = sum(v for k, v in gate_types.items()
              if k in ('XPowGate', 'YPowGate', 'FSimGate'))
dropped = sum(v for k, v in gate_types.items()
              if k not in ('XPowGate', 'YPowGate', 'FSimGate'))
print(f'\nKept (sqrt_X + sqrt_Y + FSimGate): {kept}')
print(f'Dropped (sqrt_W, Rz, other):       {dropped}')

Gate type counts in the loaded circuit:
  FSimGate: 258
  PhasedXPowGate: 226
  Rz: 1032
  XPowGate: 244
  YPowGate: 219

Kept (sqrt_X + sqrt_Y + FSimGate): 721
Dropped (sqrt_W, Rz, other):       1258


## 5. Build the 106-qubit MBQC tetron circuit

The builder below can produce either:

- `qc_mbqc`: includes final logical-output measurements, useful for optional distribution sampling;
- `qc_mbqc_nom`: omits final logical-output measurements, used for the inverse-circuit validation.

The mid-circuit MBQC measurements and feed-forward corrections are kept in both versions.


In [36]:
N_TETRON  = 106
N_DATA    = 53
N_SCRATCH = 8     # scratch bits reused by every MBQC gate call


def build_mbqc_53_circuit(qubit_order, cirq_circuit, add_final_measurements=True):
    qr        = QuantumRegister(N_TETRON,  'q')
    c_scratch = ClassicalRegister(N_SCRATCH, 'scratch')
    qc        = QuantumCircuit(qr, c_scratch)

    for moment in cirq_circuit:
        for op in moment.operations:
            g    = op.gate
            grid = [(q.row, q.col) for q in op.qubits]

            if _is_sqrt_X(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                add_sqrtX(qc, d, a, c_scratch)

            elif _is_sqrt_Y(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                add_sqrtY(qc, d, a, c_scratch)

            elif _is_fsim(g):
                # Replace any fSim(theta, phi) with fSim(pi/2, 0) = iSWAP†.
                d1, d2, anc = grid_edge_to_qiskit_indices(grid[0], grid[1])
                add_fsim_clifford(qc, d1, anc, d2, c_scratch)

            else:
                pass  # drop sqrt_W, Rz, etc.

        qc.barrier()

    if add_final_measurements:
        c_out = ClassicalRegister(N_DATA, 'out')
        qc.add_register(c_out)
        for i, q in enumerate(qubit_order):
            d = grid_to_qiskit_index(q.row, q.col)
            qc.measure(d, c_out[i])

    return qc


qc_mbqc_nom = build_mbqc_53_circuit(QUBIT_ORDER, CIRCUIT, add_final_measurements=False)
qc_mbqc     = build_mbqc_53_circuit(QUBIT_ORDER, CIRCUIT, add_final_measurements=True)

print(f'MBQC no final meas : {qc_mbqc_nom.num_qubits} qubits, '
      f'{qc_mbqc_nom.num_clbits} clbits, depth = {qc_mbqc_nom.depth()}')
print(f'MBQC with final meas: {qc_mbqc.num_qubits} qubits, '
      f'{qc_mbqc.num_clbits} clbits, depth = {qc_mbqc.depth()}')


MBQC circuit : 106 qubits, 61 clbits, depth = 78384


## 6. Build the direct 53-qubit Clifford reference circuit

This direct reference uses exactly the same Clifford projection as the tetron translation:

- `sqrt_X -> sx`
- `sqrt_Y -> S sx S†`
- `FSimGate(theta, phi) -> fSim(π/2, 0) = iSWAP†`

The builder also provides a no-final-measurement version for the inverse-circuit validation.


In [37]:
def _add_iswap_dg(qc, q0, q1):
    """iSWAP† decomposed into stabilizer-native Clifford gates.

    Derived by reversing iSWAP = S(q0)·S(q1)·H(q0)·CX(q0→q1)·CX(q1→q0)·H(q1)
    and taking the Hermitian conjugate of each factor:
        iSWAP† = H(q1)·CX(q1→q0)·CX(q0→q1)·H(q0)·Sdg(q1)·Sdg(q0)

    All instructions (h, cx, sdg) are in the Aer stabilizer gate set.
    """
    qc.h(q1)
    qc.cx(q1, q0)   # CX: q1 control, q0 target
    qc.cx(q0, q1)   # CX: q0 control, q1 target
    qc.h(q0)
    qc.sdg(q1)
    qc.sdg(q0)


def _add_sqrt_Y(qc, q):
    """sqrt(Y) = Ry(π/2) decomposed into stabilizer-native gates.

    S · SX · Sdg  =  e^{iπ/4} · √Y   (global phase irrelevant here).
    """
    qc.s(q)
    qc.sx(q)
    qc.sdg(q)


def build_direct_53_circuit(qubit_order, cirq_circuit, add_final_measurements=True):
    n          = len(qubit_order)           # 53
    qubit_map  = {q: i for i, q in enumerate(qubit_order)}
    qc         = QuantumCircuit(n, n if add_final_measurements else 0)

    for moment in cirq_circuit:
        for op in moment.operations:
            g     = op.gate
            q_idx = [qubit_map[q] for q in op.qubits]

            if   _is_sqrt_X(g):  qc.sx(q_idx[0])
            elif _is_sqrt_Y(g):  _add_sqrt_Y(qc, q_idx[0])
            elif _is_fsim(g):    _add_iswap_dg(qc, q_idx[0], q_idx[1])
            else:                pass  # dropped

        qc.barrier()

    if add_final_measurements:
        for i in range(n):
            qc.measure(i, i)

    return qc


qc_direct_nom = build_direct_53_circuit(QUBIT_ORDER, CIRCUIT, add_final_measurements=False)
qc_direct     = build_direct_53_circuit(QUBIT_ORDER, CIRCUIT, add_final_measurements=True)

print(f'Direct no final meas : {qc_direct_nom.num_qubits} qubits, '
      f'{qc_direct_nom.num_clbits} clbits, depth = {qc_direct_nom.depth()}')
print(f'Direct with final meas: {qc_direct.num_qubits} qubits, '
      f'{qc_direct.num_clbits} clbits, depth = {qc_direct.depth()}')


Direct circuit: 53 qubits, 53 clbits, depth = 100


## 7. Primary validation: inverse-direct stabilizer check

This is the scalable replacement for the 12-qubit statevector fidelity calculation.

For each MBQC measurement trajectory, we append the inverse of the direct Clifford circuit on the 53 data tetrons. If the tetron translation is correct, the measured check register should be exactly `0` repeated 53 times for every trajectory.


In [ ]:
backend_stab = AerSimulator(method='stabilizer')

# Data tetrons ordered to match the direct 53-qubit QUBIT_ORDER.
data_qargs = [grid_to_qiskit_index(q.row, q.col) for q in QUBIT_ORDER]

assert len(data_qargs) == N_DATA
assert len(set(data_qargs)) == N_DATA, 'Duplicate data tetron index detected.'
assert max(data_qargs) < N_TETRON, 'Data tetron index exceeds the 106-tetron register.'

print('Data-tetron Qiskit indices used for logical qubits:')
print(data_qargs)


In [ ]:
# Compose: MBQC trajectory followed by U_direct^† on the data tetrons.
qc_check = qc_mbqc_nom.copy()
qc_check.compose(qc_direct_nom.inverse(), qubits=data_qargs, inplace=True)

# Measure only the 53 logical/data tetrons for the final equivalence check.
c_check = ClassicalRegister(N_DATA, 'check')
qc_check.add_register(c_check)
for i, q in enumerate(data_qargs):
    qc_check.measure(q, c_check[i])

print(f'Check circuit: {qc_check.num_qubits} qubits, '
      f'{qc_check.num_clbits} clbits, depth = {qc_check.depth()}')


In [ ]:
N_TRAJECTORIES = 20   # each shot is one MBQC measurement trajectory
SEED_CHECK     = 1234

print('Transpiling check circuit for Aer stabilizer backend...')
t0 = time.perf_counter()
qc_check_t = transpile(qc_check, backend_stab, optimization_level=0)
t1 = time.perf_counter()
print(f'Transpile time: {t1 - t0:.3f} s')

print(f'Running {N_TRAJECTORIES} MBQC trajectories...')
t0 = time.perf_counter()
result_check = backend_stab.run(
    qc_check_t,
    shots=N_TRAJECTORIES,
    seed_simulator=SEED_CHECK,
).result()
t1 = time.perf_counter()

counts_check_raw = result_check.get_counts()
print(f'Run time: {t1 - t0:.3f} s')
print('Raw counts including scratch/check registers:')
print(counts_check_raw)


In [ ]:
def extract_leftmost_register(counts_combined):
    """Return counts for the leftmost space-separated classical register.

    Qiskit prints multiple classical registers as space-separated blocks. Since
    the check register was added after scratch, it is normally the leftmost block.
    The all-zero success condition is insensitive to bit order inside this block.
    """
    out = {}
    for key, val in counts_combined.items():
        reg_bits = key.split()[0]
        out[reg_bits] = out.get(reg_bits, 0) + val
    return out


counts_check = extract_leftmost_register(counts_check_raw)
zero = '0' * N_DATA
zero_count = counts_check.get(zero, 0)

print('Check-register counts only:')
print(counts_check)
print(f'\nAll-zero trajectories: {zero_count} / {N_TRAJECTORIES}')

if zero_count == N_TRAJECTORIES and len(counts_check) == 1:
    print('PASS: every MBQC trajectory matches the direct Clifford circuit.')
else:
    print('FAIL / DEBUG NEEDED: at least one trajectory did not return to |0...0>.')
    print('Nonzero check strings:')
    for bitstr, count in sorted(counts_check.items(), key=lambda kv: -kv[1]):
        if bitstr != zero:
            print(bitstr, count)


## 9. Notes and caveats

### Why not direct statevectors?

The 12-qubit notebook computed exact trajectory-by-trajectory fidelity from full statevectors. With 53 logical qubits, a direct statevector has \(2^{53}\) complex amplitudes; the tetron MBQC Hilbert space would be even larger because it contains 106 tetrons before tracing out ancillas. This is not practical.

### Why not use sampled Hellinger fidelity as the main metric?

For a 53-qubit random-circuit-like output distribution, the support is enormous. Even if two circuits are identical, two independent samples with realistic shot counts will overlap only weakly as histograms. Thus finite-shot Hellinger fidelity is a poor way to certify equality here.

### What the inverse-check certifies

For the Clifford-projected circuit, the stabilizer backend can propagate every MBQC measurement trajectory efficiently. If the feed-forward rules and tetron mapping are correct, appending the inverse direct Clifford circuit maps every trajectory back to \(|0\rangle^{\otimes 53}\) on the data tetrons. This gives a sharp pass/fail test without constructing a statevector and without estimating the full output distribution.

### Clifford approximation made

The actual `FSimGate` angles in the Google file are calibrated values with \(\theta\) near \(\pi/2\) and \(\phi\) near \(\pi/6\). This notebook replaces all of them with `fSim(π/2, 0) = iSWAP†`, and drops `Rz` and `PhasedXPowGate` gates. Therefore this notebook validates the MBQC translation machinery for the Clifford-projected circuit.

### Mapping file

The notebook prefers `qubit_mapping_53_connected_47_51.py`, where logical labels 41 and 47 are swapped so the 47--51 edge is locally connected through an ancilla. If that file is not available, it falls back to `qubit_mapping_53.py`.
